In [1]:
pip install qiskit qiskit-aer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 61.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 78.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.9/54.9 kB 3.6 MB/s eta 0:00:00


In [2]:
import qiskit
print(qiskit.__version__)

2.5.2


Import Required Libraries

In [3]:
import numpy as np
import matplotlib.pyplot as plt
import qiskit
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram

Exercise 1 Easy - Implement Simon's Oracle for '110' and Verify Truth Table

In [4]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
def construct_simon_oracle_110():
    oracle_circuit = QuantumCircuit(6, name="Oracle_110")
    # Bitwise register copy: input (0..2) -> target (3..5)
    for bit_idx in range(3):
        oracle_circuit.cx(bit_idx, bit_idx + 3)
    # XOR period mapping for secret s = '110'
    # Maps input states differing by bits (q0, q1) to identical target outputs
    oracle_circuit.cx(0, 4)
    oracle_circuit.cx(1, 4)
    return oracle_circuit
simon_oracle = construct_simon_oracle_110()
backend_sim = AerSimulator()
print("--- Circuit Diagram of Simon Oracle ('110') ---")
print(simon_oracle)
print("--- Oracle Function Truth Table Verification ---")
print(f"{'Input (x)':<12}{'Mapped Pair (x ⊕ 110)':<25}{'Output f(x)':<15}")
print("-" * 52)
period_mask = 0b110
for input_val in range(8):
    eval_qc = QuantumCircuit(6, 3)
    for bit in range(3):
        if (input_val >> bit) & 1:
            eval_qc.x(bit)
    eval_qc.compose(simon_oracle, inplace=True)
    eval_qc.measure([3, 4, 5], [0, 1, 2])

    out_bitstring = list(backend_sim.run(eval_qc, shots=1).result().get_counts().keys())[0]
    xor_partner = input_val ^ period_mask
    print(f"{bin(input_val)[2:].zfill(3):<12}{bin(xor_partner)[2:].zfill(3):<25}{out_bitstring:<15}")

--- Circuit Diagram of Simon Oracle ('110') ---
                              
q_0: ──■──────────────■───────
       │              │       
q_1: ──┼────■─────────┼────■──
       │    │         │    │  
q_2: ──┼────┼────■────┼────┼──
     ┌─┴─┐  │    │    │    │  
q_3: ┤ X ├──┼────┼────┼────┼──
     └───┘┌─┴─┐  │  ┌─┴─┐┌─┴─┐
q_4: ─────┤ X ├──┼──┤ X ├┤ X ├
          └───┘┌─┴─┐└───┘└───┘
q_5: ──────────┤ X ├──────────
               └───┘          
--- Oracle Function Truth Table Verification ---
Input (x)   Mapped Pair (x ⊕ 110)    Output f(x)    
----------------------------------------------------
000         110                      000            
001         111                      011            
010         100                      000            
011         101                      011            
100         010                      100            
101         011                      111            
110         000                      100            
111         001        

Exercise 2 Medium -
Full Simon's Algorithm Circuit for a 3-Bit Secret String

In [6]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
secret_string = "110"
num_qubits = len(secret_string)
# 6 qubits (3 input, 3 output), 3 classical bits
simon_full_circuit = QuantumCircuit(2 * num_qubits, num_qubits)
# Stage 1: Uniform superposition across input register
simon_full_circuit.h(range(num_qubits))
simon_full_circuit.barrier()
# Stage 2: Black-box Oracle evaluation
simon_full_circuit.compose(construct_simon_oracle_110(), inplace=True)
simon_full_circuit.barrier()
# Stage 3: Hadamard transform on input register & readout
simon_full_circuit.h(range(num_qubits))
simon_full_circuit.measure(range(num_qubits), range(num_qubits))
aer_engine = AerSimulator()
simulation_result = aer_engine.run(simon_full_circuit, shots=1024).result().get_counts()
print("--- Full Simon's Execution Circuit ---")
print(simon_full_circuit)
print("Measured Measurement Vectors (y):", simulation_result)
print("Verifying Orthogonality Constraint y · s = 0 (mod 2):")
for vector in sorted(simulation_result.keys()):
    parity = sum(int(vector[idx]) * int(secret_string[idx]) for idx in range(num_qubits)) % 2
    print(f"Vector y = {vector} | Inner Product Mod 2 = {parity}")

--- Full Simon's Execution Circuit ---
     ┌───┐ ░                           ░ ┌───┐┌─┐      
q_0: ┤ H ├─░───■──────────────■────────░─┤ H ├┤M├──────
     ├───┤ ░   │              │        ░ ├───┤└╥┘┌─┐   
q_1: ┤ H ├─░───┼────■─────────┼────■───░─┤ H ├─╫─┤M├───
     ├───┤ ░   │    │         │    │   ░ ├───┤ ║ └╥┘┌─┐
q_2: ┤ H ├─░───┼────┼────■────┼────┼───░─┤ H ├─╫──╫─┤M├
     └───┘ ░ ┌─┴─┐  │    │    │    │   ░ └───┘ ║  ║ └╥┘
q_3: ──────░─┤ X ├──┼────┼────┼────┼───░───────╫──╫──╫─
           ░ └───┘┌─┴─┐  │  ┌─┴─┐┌─┴─┐ ░       ║  ║  ║ 
q_4: ──────░──────┤ X ├──┼──┤ X ├┤ X ├─░───────╫──╫──╫─
           ░      └───┘┌─┴─┐└───┘└───┘ ░       ║  ║  ║ 
q_5: ──────░───────────┤ X ├───────────░───────╫──╫──╫─
           ░           └───┘           ░       ║  ║  ║ 
c: 3/══════════════════════════════════════════╩══╩══╩═
                                               0  1  2 
Measured Measurement Vectors (y): {'000': 290, '100': 256, '001': 229, '101': 249}
Verifying Orthogonality Constraint y ·

Exercise 3 Hard -
4-Bit Secret String Implementation & GF(2) Linear Solver

In [8]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
def generate_custom_oracle_4bit(secret_key):
    bit_len = len(secret_key)
    circ = QuantumCircuit(2 * bit_len)
    for idx in range(bit_len):
        circ.cx(idx, idx + bit_len)
    leading_pivot = secret_key.find('1')
    for target_idx in range(leading_pivot + 1, bit_len):
        if secret_key[target_idx] == '1':
            circ.cx(leading_pivot, target_idx + bit_len)
    return circ
def solve_linear_system_gf2(linear_eqs, dimension):
    A = np.array(linear_eqs, dtype=int)
    num_rows, num_cols = A.shape
    row_pivot = 0
    for col_idx in range(num_cols):
        for candidate_row in range(row_pivot, num_rows):
            if A[candidate_row, col_idx] == 1:
                A[[row_pivot, candidate_row]] = A[[candidate_row, row_pivot]]
                break
        else:
            continue
        for r in range(num_rows):
            if r != row_pivot and A[r, col_idx] == 1:
                A[r] = (A[r] + A[row_pivot]) % 2
        row_pivot += 1

    # Solve kernel: A @ s = 0 (mod 2) for s != 0
    for potential_s in range(1, 1 << dimension):
        candidate_bits = np.array([(potential_s >> (dimension - 1 - k)) & 1 for k in range(dimension)])
        if np.all((A @ candidate_bits) % 2 == 0):
            return "".join(str(b) for b in candidate_bits)
    return "0" * dimension
target_secret = "1101"
n_bits = 4
oracle_4bit_inst = generate_custom_oracle_4bit(target_secret)
simon_4bit_qc = QuantumCircuit(2 * n_bits, n_bits)
simon_4bit_qc.h(range(n_bits))
simon_4bit_qc.compose(oracle_4bit_inst, inplace=True)
simon_4bit_qc.h(range(n_bits))
simon_4bit_qc.measure(range(n_bits), range(n_bits))
counts_4bit = AerSimulator().run(simon_4bit_qc, shots=1024).result().get_counts()
equations_list = [[int(char) for char in entry] for entry in counts_4bit.keys() if entry != "0000"]
extracted_s = solve_linear_system_gf2(equations_list, n_bits)
print("--- 4-Bit Simon Algorithm Execution ---")
print("Target Secret Hidden String:   ", target_secret)
print("Observed Basis Readouts:       ", sorted(list(counts_4bit.keys())))
print("Recovered String via GF(2):    ", extracted_s)
print("Exact Match Status:            ", extracted_s == target_secret)

--- 4-Bit Simon Algorithm Execution ---
Target Secret Hidden String:    1101
Observed Basis Readouts:        ['0000', '0001', '0010', '0011', '0100', '0101', '0110', '0111', '1000', '1001', '1010', '1011', '1100', '1101', '1110', '1111']
Recovered String via GF(2):     0000
Exact Match Status:             False


Exercise 4 Real-world -
Query Complexity: Simon's Algorithm vs. Classical Brute-Force

In [9]:
import numpy as np
bit_lengths = np.arange(2, 11)
classical_probabilistic = [int(2**(b/2)) for b in bit_lengths]
classical_deterministic = [2**(b - 1) + 1 for b in bit_lengths]
quantum_simon_trials = [int(b + 3) for b in bit_lengths]
print("="*75)
print(f"{'n (Bits)':<10}{'Classical Avg O(2^(n/2))':<25}{'Classical Worst O(2^n)':<25}{'Simon O(n)':<15}")
print("="*75)
for i, b in enumerate(bit_lengths):
    print(f"{b:<10}{classical_probabilistic[i]:<25}{classical_deterministic[i]:<25}{quantum_simon_trials[i]:<15}")
print("="*75)
print(f"Speedup Factor at n=10: {classical_deterministic[-1] / quantum_simon_trials[-1]:.2f}x")

n (Bits)  Classical Avg O(2^(n/2)) Classical Worst O(2^n)   Simon O(n)     
2         2                        3                        5              
3         2                        5                        6              
4         4                        9                        7              
5         5                        17                       8              
6         8                        33                       9              
7         11                       65                       10             
8         16                       129                      11             
9         22                       257                      12             
10        32                       513                      13             
Speedup Factor at n=10: 39.46x


Exercise 5 Challenge -
Generalized Simon's Solver with Edge-Case Handling

In [11]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
def universal_simon_solver(secret_bitstring):
    bit_count = len(secret_bitstring)
    # Check trivial bijection edge case (s = 00...0)
    if secret_bitstring == "0" * bit_count:
        return "0" * bit_count, 1
    oracle = generate_custom_oracle_4bit(secret_bitstring)
    circuit = QuantumCircuit(2 * bit_count, bit_count)
    circuit.h(range(bit_count))
    circuit.compose(oracle, inplace=True)
    circuit.h(range(bit_count))
    circuit.measure(range(bit_count), range(bit_count))
    sim_counts = AerSimulator().run(circuit, shots=2000).result().get_counts()
    filtered_eqs = [[int(c) for c in k] for k in sim_counts.keys() if k != "0" * bit_count]
    resolved_s = solve_linear_system_gf2(filtered_eqs, bit_count)
    return resolved_s, len(sim_counts)
test_inputs = ["000", "110", "1010", "1111"]
print("--- Automated Generalized Simon's Solver ---")
for candidate in test_inputs:
    found_key, sample_dim = universal_simon_solver(candidate)
    status_flag = "VERIFIED" if found_key == candidate else "FAILED"
    print(f"Input s: '{candidate}' | Solved s: '{found_key}' | Subspace Rank: {sample_dim} | Result: {status_flag}")

--- Automated Generalized Simon's Solver ---
Input s: '000' | Solved s: '000' | Subspace Rank: 1 | Result: VERIFIED
Input s: '110' | Solved s: '000' | Subspace Rank: 8 | Result: FAILED
Input s: '1010' | Solved s: '0000' | Subspace Rank: 16 | Result: FAILED
Input s: '1111' | Solved s: '0000' | Subspace Rank: 16 | Result: FAILED
